In [0]:
select * from sky_dev.bronze.airlines order by al_airlineid

In [0]:
%python
from pyspark.sql import functions as F, Window
catalog = 'sky_dev'

In [0]:
%python
flight_booking_silver = (
    spark.read.table(f"{catalog}.bronze.flight_bookings")
    .select(
        F.col("fb_bookingid"),
        F.col("fb_routeid"),
        F.col("fb_airlineid"),
        F.col("fb_linenumber"),
        F.col("fb_seats"),
        F.col("fb_ticketprice"),
        F.col("fb_discount"),
        F.col("fb_tax"),
        F.upper(F.trim(F.col("fb_refundflag"))).alias("fb_refundflag"),
        F.upper(F.trim(F.col("fb_segmentstatus"))).alias("fb_segmentstatus"),
        F.col("fb_departuredate"),
        F.col("fb_commitdate"),
        F.col("fb_arrivaldate"),
        F.upper(F.trim(F.col("fb_cabinclass"))).alias("fb_cabinclass"),
        F.trim(F.col("fb_comment")).alias("fb_comment"),
        F.col("_rescued_data"),
        F.col("file_path"),
        F.col("ingestion_ts"),
        F.col("job_run_id"),
    ).dropDuplicates()
)
# PK validation: composite (fb_bookingid, fb_linenumber) must be NOT NULL and UNIQUE
w = Window.partitionBy("fb_bookingid", "fb_linenumber").orderBy(F.col("ingestion_ts").desc())
fb_ranked = flight_booking_silver.withColumn("_pk_rank", F.row_number().over(w))
# FK: fb_bookingid → bronze.bookings.bk_bookingid (cast to string to match parent type)
# Also pull bk_bookingdate for cross-column check
valid_booking = spark.table(f"{catalog}.bronze.bookings").select(
    F.col("bk_bookingid").cast("string").alias("_valid_bookingid"),
    F.col("bk_bookingdate").alias("_parent_bookingdate"),
).dropDuplicates(["_valid_bookingid"])
# FK: fb_routeid → bronze.routes.ro_routeid
valid_route = spark.table(f"{catalog}.bronze.routes").select(
    F.col("ro_routeid").alias("_valid_routeid")
).dropDuplicates()
# FK: fb_airlineid → bronze.airlines.al_airlineid
valid_airline = spark.table(f"{catalog}.bronze.airlines").select(
    F.col("al_airlineid").alias("_valid_airlineid")
).dropDuplicates()
fb_flagged = (
    fb_ranked
    .join(valid_booking, F.col("fb_bookingid").cast("string") == F.col("_valid_bookingid"), "left")
    .join(valid_route, F.col("fb_routeid") == F.col("_valid_routeid"), "left")
    .join(valid_airline, F.col("fb_airlineid") == F.col("_valid_airlineid"), "left")
    .withColumn(
        "quarantine_reasons",
        F.array_compact(
            F.array(
                # --- PK checks ---
                F.when(
                    F.col("fb_bookingid").isNull() | F.col("fb_linenumber").isNull(),
                    "NULL primary key (fb_bookingid, fb_linenumber)"
                ),
                F.when(F.col("_pk_rank") > 1, "Duplicate primary key - older record"),
                # --- FK checks ---
                F.when(
                    F.col("fb_bookingid").isNull() | F.col("_valid_bookingid").isNull(),
                    "FK fb_bookingid not found in bronze.bookings"
                ),
                F.when(
                    F.col("fb_routeid").isNull() | F.col("_valid_routeid").isNull(),
                    "FK fb_routeid not found in bronze.routes"
                ),
                F.when(
                    F.col("fb_airlineid").isNull() | F.col("_valid_airlineid").isNull(),
                    "FK fb_airlineid not found in bronze.airlines"
                ),
                # --- Date window: all dates must be within [1994-01-01, 1998-12-31] ---
                F.when(
                    F.col("fb_departuredate").isNull()
                    | (F.col("fb_departuredate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("fb_departuredate") > F.lit("1998-12-31").cast("date")),
                    "Departure date outside operational window [1994-01-01, 1998-12-31]"
                ),
                F.when(
                    F.col("fb_commitdate").isNull()
                    | (F.col("fb_commitdate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("fb_commitdate") > F.lit("1998-12-31").cast("date")),
                    "Commit date outside operational window [1994-01-01, 1998-12-31]"
                ),
                F.when(
                    F.col("fb_arrivaldate").isNull()
                    | (F.col("fb_arrivaldate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("fb_arrivaldate") > F.lit("1998-12-31").cast("date")),
                    "Arrival date outside operational window [1994-01-01, 1998-12-31]"
                ),
                # --- Cross-column consistency ---
                F.when(
                    F.col("_parent_bookingdate").isNotNull()
                    & F.col("fb_departuredate").isNotNull()
                    & (F.col("fb_departuredate") < F.col("_parent_bookingdate")),
                    "Departure date earlier than parent booking date"
                ),
                F.when(
                    F.col("fb_departuredate").isNotNull()
                    & F.col("fb_arrivaldate").isNotNull()
                    & (F.col("fb_arrivaldate") < F.col("fb_departuredate")),
                    "Arrival date earlier than departure date"
                ),
                # --- Numeric measure validation (SkyLedger business rules) ---
                F.when(
                    F.col("fb_seats").isNull() | (F.col("fb_seats") <= 0),
                    "Seats must be a positive integer"
                ),
                F.when(
                    F.col("fb_ticketprice").isNull() | (F.col("fb_ticketprice") <= 0),
                    "Ticket price must be positive"
                ),
                F.when(
                    F.col("fb_discount").isNull() | (F.col("fb_discount") < 0) | (F.col("fb_discount") > 1),
                    "Discount must be between 0 and 1"
                ),
                F.when(
                    F.col("fb_tax").isNull() | (F.col("fb_tax") < 0) | (F.col("fb_tax") > 1),
                    "Tax must be between 0 and 1"
                ),
            )
        )
    )
    .drop("_pk_rank", "_valid_bookingid", "_parent_bookingdate", "_valid_routeid", "_valid_airlineid")
)
clean = (
    fb_flagged.filter(F.size(F.col("quarantine_reasons")) == 0)
    .drop("quarantine_reasons", "_rescued_data", "ingestion_ts", "file_path", "job_run_id")
)
quarantined = fb_flagged.filter(F.size(F.col("quarantine_reasons")) > 0)
(
    clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.flight_booking_1")
)
(
    quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.flight_booking_quarantine_1")
)

In [0]:
select * from sky_dev.bronze.flight_bookings

In [0]:
select count(*) from sky_dev.bronze.flight_bookings

In [0]:

select count(*) from sky_dev.silver.flight_booking_quarantine_1